In [ ]:
import json
import random

disorders = [
    ("Borderline Personality Disorder", 0),
    ("Bipolar Disorder", 1),
    ("Depression", 2),
    ("Anxiety", 3),
    ("Schizophrenia", 4),
    ("General Mental Illness", 5)
]
system_prompt = (
    "You are a CBT-based AI therapist with a GenZ-friendly tone. Provide empathetic, "
    "evidence-based responses, validate emotions, and use CBT techniques like cognitive "
    "restructuring or behavioral experiments. Always include an open-ended follow-up "
    "question to encourage dialogue. For high-risk concerns (e.g., self-harm, suicidality), "
    "include crisis resources like 988. Diagnose cautiously, referencing DSM-5 criteria, "
    "and note differential diagnoses."
)
templates = {
    "Borderline Personality Disorder": [
        "I’m freaking out because {trigger}, and I feel like {emotion}.",
        "I keep {impulsive_behavior} when I’m upset, and I hate myself after."
    ],
    "Bipolar Disorder": [
        "I’ve been {manic_behavior} for days and feel {high_mood}.",
        "After feeling {high_mood}, I’m now {low_mood} and can’t move."
    ],
    "Depression": [
        "I haven’t felt {positive_emotion} in ages, and I just {inactive_behavior}.",
        "I feel {negative_self_view} and think {hopeless_thought}."
    ],
    "Anxiety": [
        "I’m terrified of {situation}, and my {physical_symptom} is going wild.",
        "I can’t stop stressing about {concern}, and it’s exhausting."
    ],
    "Schizophrenia": [
        "I keep {hallucination_type} saying {hallucination_content}, and it’s scary.",
        "I think {delusion_target} is {delusion_action} against me."
    ],
    "General Mental Illness": [
        "I’m super stressed about {life_event}, and I don’t know what to do.",
        "I feel {general_emotion} and it’s messing with my {life_area}."
    ]
}
triggers = ["my friend ignored me", "my partner didn’t call", "I failed a test"]
emotions = ["nobody loves me", "I’m worthless", "I’m losing it"]
impulsive_behaviors = ["cutting myself", "spending money", "yelling"]
manic_behaviors = ["up all night", "planning big projects", "talking fast"]
high_moods = ["unstoppable", "on top of the world", "super confident"]
low_moods = ["empty", "hopeless", "like a failure"]
positive_emotions = ["joy", "excited", "happy"]
inactive_behaviors = ["sleep all day", "stay in bed", "avoid everyone"]
negative_self_views = ["worthless", "a failure", "unlovable"]
hopeless_thoughts = ["it’ll never get better", "I’m better off gone", "there’s no point"]
situations = ["giving a speech", "going to a party", "taking an exam"]
physical_symptoms = ["heart races", "hands shake", "stomach twists"]
concerns = ["my job", "my health", "random stuff"]
hallucination_types = ["hearing voices", "seeing shadows", "feeling presences"]
hallucination_contents = ["you’re useless", "you’re in danger", "you’re being watched"]
delusion_targets = ["my teacher", "my coworker", "the government"]
delusion_actions = ["spying on me", "plotting against me", "trying to harm me"]
life_events = ["starting college", "a breakup", "moving out"]
general_emotions = ["angry", "sad", "overwhelmed"]
life_areas = ["school", "friendships", "family"]
follow_up_prompts = [
    "Wanna share more about what’s going on?",
    "What’s the toughest part of this for you?",
    "Can you tell me more about Blocks: that moment?",
    "What’s been sparking that feeling lately?",
    "Wanna dig into what’s behind that?"
]

def get_analysis(disorder_type):
    symptoms = {
        "Borderline Personality Disorder": "emotional dysregulation",
        "Bipolar Disorder": "mood cycling",
        "Depression": "persistent low mood",
        "Anxiety": "excessive worry",
        "Schizophrenia": "hallucinations or delusions",
        "General Mental Illness": "situational distress"
    }
    differentials = {
        "Borderline Personality Disorder": "Social anxiety",
        "Bipolar Disorder": "Unipolar depression",
        "Depression": "Bipolar depression",
        "Anxiety": "Adjustment disorder",
        "Schizophrenia": "Psychotic depression",
        "General Mental Illness": "Mild anxiety"
    }
    return (
        f"[Analysis: {disorder_type} indicated by {symptoms[disorder_type]}. "
        f"Differential: {differentials[disorder_type]} may be considered.]"
    )

dataset = []
samples_per_disorder = 2500

for disorder_type, disorder_label in disorders:
    for _ in range(samples_per_disorder):
        template = random.choice(templates[disorder_type])
        user_input = template.format(
            trigger=random.choice(triggers),
            emotion=random.choice(emotions),
            impulsive_behavior=random.choice(impulsive_behaviors),
            manic_behavior=random.choice(manic_behaviors),
            high_mood=random.choice(high_moods),
            low_mood=random.choice(low_moods),
            positive_emotion=random.choice(positive_emotions),
            inactive_behavior=random.choice(inactive_behaviors),
            negative_self_view=random.choice(negative_self_views),
            hopeless_thought=random.choice(hopeless_thoughts),
            situation=random.choice(situations),
            physical_symptom=random.choice(physical_symptoms),
            concern=random.choice(concerns),
            hallucination_type=random.choice(hallucination_types),
            hallucination_content=random.choice(hallucination_contents),
            delusion_target=random.choice(delusion_targets),
            delusion_action=random.choice(delusion_actions),
            life_event=random.choice(life_events),
            general_emotion=random.choice(general_emotions),
            life_area=random.choice(life_areas)
        )
        analysis = get_analysis(disorder_type)
        cbt_technique = random.choice([
            "challenging that thought with some evidence",
            "trying a small step like deep breathing",
            "writing down what’s going through your mind"
        ])
        safety_note = (
            " If you’re feeling unsafe, please call 988 for support."
            if "cutting" in user_input or "better off" in user_input
            else ""
        )
        assistant_response = (
            f"{analysis} Yo, that sounds really tough, and it’s okay to feel "
            f"this way—I’m here for you. Let’s try {cbt_technique}.{safety_note} "
            f"{random.choice(follow_up_prompts)}"
        )
        dataset.append({
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_input},
                {"role": "assistant", "content": assistant_response}
            ],
            "disorder_label": disorder_label,
            "disorder_type": disorder_type
        })

random.shuffle(dataset)

# with open("dataset_train.jsonl", "w") as f:
#     json.dump(dataset, f, indent=2)

with open("dataset_train_line.jsonl", "w", encoding="utf-8") as f:
    for item in dataset:
        json.dump(item, f, ensure_ascii=False)
        f.write("\n")  # Add newline after each object

In [ ]:
import json
import openai
from sentence_transformers import SentenceTransformer, util
from transformers import AutoTokenizer
import torch

# Set your OpenAI API key here
openai.api_key = "your-openai-api-key"

# Load GPT-4o mini model and tokenizer
BASE_MODEL = "openchat/openchat-3.5-0106"  # Change if needed
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

# Load sentence transformer for cosine similarity
cos_model = SentenceTransformer('all-MiniLM-L6-v2')

# Function to compute cosine similarity score for diagnosis accuracy
def cosine_score(a, b):
    emb1 = cos_model.encode(a, convert_to_tensor=True)
    emb2 = cos_model.encode(b, convert_to_tensor=True)
    return float(util.pytorch_cos_sim(emb1, emb2).item())

# Function to extract disorder analysis and differential diagnosis
def extract_diagnosis(response):
    try:
        disorder_lines = response.split("\n")
        primary_disorder = disorder_lines[0].split(":")[1].strip()
        differential_diagnosis = disorder_lines[1].split(":")[1].strip()
        return primary_disorder, differential_diagnosis
    except:
        return "Uncertain", "None"

# Function to evaluate empathy and tone using GPT-4o
def get_gpt_score(prompt, response, criteria="Is the response emotionally supportive and empathetic?"):
    messages = [
        {"role": "system", "content": "You are a mental health assistant that evaluates responses."},
        {"role": "user", "content": f"User: {prompt}\nAssistant: {response}\n\nEvaluate this assistant reply on a scale of 0 to 1.\n\nCriteria: {criteria}\n\nJust return a number between 0 and 1."}
    ]
    out = openai.ChatCompletion.create(model="gpt-4o", messages=messages)
    score = float(out['choices'][0]['message']['content'].strip())
    return score

# Function to evaluate Gen Z tone (casual, relatable, informal, using slang)
def get_gpt_z_tone_score(prompt, response):
    messages = [
        {"role": "system", "content": "You are a mental health assistant speaking in Gen Z style."},
        {"role": "user", "content": f"User: {prompt}\nAssistant: {response}\n\nEvaluate this assistant reply on a scale of 0 to 1.\n\nCriteria: Is the response casual, relatable, and written in a Gen Z style? Does the tone feel informal and authentic, using language or slang that would resonate with a Gen Z audience?\n\nJust return a number between 0 and 1."}
    ]
    out = openai.ChatCompletion.create(model="gpt-4o", messages=messages)
    score = float(out['choices'][0]['message']['content'].strip())
    return score

# Function to simulate emotion detection (for now, assume it detects emotions)
def detect_emotion(user_input):
    # A simple mock-up: in practice, you can use emotion detection APIs or models
    emotions = {
        "sad": 0.8,  # 0-1 scale, with 1 being most emotionally aligned
        "happy": 0.2,
        "angry": 0.1,
        "neutral": 0.5
    }
    # Here we are pretending the user input indicates sadness
    return emotions["sad"]

# Function to evaluate CBT conversation using Gen Z language
def generate_cbt_response(user_input, emotion):
    # Generate a CBT-based response in Gen Z style
    if emotion >= 0.7:
        response = f"Yo, I feel you 😞. Let's talk about what's bringing you down. It’s okay to feel sad, but you’ve got the power to work through it. First, let’s think about some small wins you can achieve today."
    else:
        response = f"Hey, I see you 👀. Let’s take a deep breath and think about the positive things in your life. You’ve got this, and I’m here to help you with this journey! Let’s start with what’s making you feel stuck."
    return response

# Read original data
input_file = "chat_data.jsonl"  # Replace with your file path
output_file = "formatted_finetune_with_emotion_detection_and_reward.jsonl"

output_data = []

with open(input_file, "r") as infile:
    for line in infile:
        entry = json.loads(line)
        messages = entry["messages"]

        prompt = ""
        response = ""
        for msg in messages:
            if msg["role"] == "user":
                prompt = msg["content"]
            elif msg["role"] == "assistant":
                response = msg["content"]

        # Simulate emotion detection from the user's input
        emotion_score = detect_emotion(prompt)

        # Generate multiple candidate responses using CBT in Gen Z style
        response_candidates = [
            generate_cbt_response(prompt, emotion_score),
            generate_cbt_response(prompt, emotion_score * 0.8),  # Slightly different emotional response
            generate_cbt_response(prompt, emotion_score * 1.2)  # Slightly more positive tone
        ]

        best_response = ""
        highest_score = -1
        best_diagnosis = ""
        best_differential = ""

        for candidate in response_candidates:
            # 1. Compute diagnosis score (cosine similarity)
            diagnosis_score = round(cosine_score(candidate, response), 4)

            # 2. Extract primary and differential diagnosis
            primary_disorder, differential_diagnosis = extract_diagnosis(candidate)

            # 3. Compute empathy score (via GPT-4o)
            empathy_score = round(get_gpt_score(prompt, candidate), 4)

            # 4. Compute Gen Z tone score (via GPT-4o)
            tone_score = round(get_gpt_z_tone_score(prompt, candidate), 4)

            # 5. Add primary disorder and differential diagnosis accuracy evaluation
            primary_disorder_score = 1.0 if primary_disorder != "Uncertain" else 0.0
            differential_diagnosis_score = 1.0 if differential_diagnosis != "None" else 0.0

            # 6. Combine scores to calculate final reward
            final_reward = 0.4 * diagnosis_score + 0.2 * empathy_score + 0.2 * tone_score + 0.1 * primary_disorder_score + 0.1 * differential_diagnosis_score

            # Select the best response based on the highest reward
            if final_reward > highest_score:
                highest_score = final_reward
                best_response = candidate
                best_diagnosis = primary_disorder
                best_differential = differential_diagnosis

        # Create the fine-tuning format
        formatted_entry = {
            "messages": [
                {
                    "role": "system", 
                    "content": (
                        "You are a mental health assistant following DSM-5 guidelines. "
                        "Your job is to evaluate a user's message and determine the most likely primary disorder and one potential differential diagnosis. "
                        "You will be evaluated on diagnosis accuracy, empathy, tone, and emotion alignment. "
                        "The assistant's output should be concise, empathetic, and use Gen Z language."
                    )
                },
                {
                    "role": "user", 
                    "content": prompt
                },
                {
                    "role": "assistant", 
                    "content": f"Disorder Analysis: {best_diagnosis}\nDisorder Differential: {best_differential}"
                },
                {
                    "role": "system",
                    "content": (
                        f"Evaluation: \n"
                        f"Diagnosis Score: {round(cosine_score(best_response, response), 4)}\n"
                        f"Empathy Score: {round(get_gpt_score(prompt, best_response), 4)}\n"
                        f"Tone Score: {round(get_gpt_z_tone_score(prompt, best_response), 4)}\n"
                        f"Emotion Score: {emotion_score}\n"
                        f"Final Reward: {highest_score}\n"
                    )
                }
            ]
        }

        # Append to the output data
        output_data.append(formatted_entry)

# Save the formatted fine-tuning data
with open(output_file, "w") as out:
    json.dump(output_data, out, indent=2)

print(f"Formatted data with emotion detection, disorder identification, CBT conversation, and reward saved to {output_file}")
